<a href="https://colab.research.google.com/github/BNWPMBA/BAN5600/blob/main/Live_Dashboard_Test.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Google Sheets Authentication Setup

To access Google Sheets from Colab, you need to authenticate your Colab environment with your Google account. This usually involves:

1.  **Enabling the Google Sheets API**: Make sure the Google Sheets API is enabled for your Google Cloud Project. If you're using a personal Google account, this is often handled automatically or through a consent screen.
2.  **Using `colab.auth`**: Colab provides a convenient way to authenticate using `google.colab.auth` which handles the OAuth2 flow.

First, let's install the necessary libraries.

In [67]:
# Install necessary libraries
!pip install --upgrade gspread pandas

In [68]:
import gspread
import pandas as pd
from google.colab import auth
import google.auth

# Authenticate Google Colab
auth.authenticate_user()

# Get default credentials from the authenticated user
credentials, project = google.auth.default(
    scopes=[
        'https://www.googleapis.com/auth/spreadsheets',
        'https://www.googleapis.com/auth/drive'
    ]
)

# Authorize gspread with the authenticated user's credentials
gc = gspread.Client(auth=credentials)

print("Authentication complete. gspread client ready.")

Authentication complete. gspread client ready.


Now that we're authenticated, you can open a Google Sheet by its name or URL and read its data. Replace the placeholder URL with your actual Google Sheet URL.

In [69]:
# Replace with your Google Sheet URL
sheet_url = 'https://docs.google.com/spreadsheets/d/1MWQECkBsbjz6AaP7uwCuEFR6xMyzYxy9z6JrOkjGPKM/edit?gid=0#gid=0'

# Open the sheet by URL
try:
    worksheet = gc.open_by_url(sheet_url).sheet1 # .sheet1 refers to the first worksheet
    print(f"Successfully opened sheet: {worksheet.title}")

    # Get all values from the worksheet as a list of lists
    data = worksheet.get_all_values()

    # Convert to a pandas DataFrame
    # The first row is usually the header
    df = pd.DataFrame(data[1:], columns=data[0])

    print("\nFirst 5 rows of the DataFrame:")
    display(df.head())

except gspread.exceptions.SpreadsheetNotFound:
    print(f"Error: Spreadsheet not found at URL: {sheet_url}. Please ensure the URL is correct and you have access.")
except Exception as e:
    print(f"An error occurred: {e}")

Successfully opened sheet: Sheet1

First 5 rows of the DataFrame:


,Claim Id,Samples Received,Type,Eval Time
0,1,TRUE,Standard,11
1,2,TRUE,Standard,11
2,3,TRUE,Standard,9
3,4,TRUE,Standard,8
4,5,TRUE,Extended,8


### Data Cleaning and Transformation

First, let's ensure the 'Eval Time' column is numeric and then apply the type transformation as requested.

In [70]:
# Convert 'Eval Time' to numeric, coercing errors to NaN
df['Eval Time'] = pd.to_numeric(df['Eval Time'], errors='coerce')

# Fill any NaNs that might result from coercion with a default (e.g., 0) or handle them as appropriate
df['Eval Time'] = df['Eval Time'].fillna(0).astype(int)

# Convert all 'Type' values that are not 'NMC' to 'Standard'
df.loc[df['Type'] != 'NMC', 'Type'] = 'Standard'

print("DataFrame after 'Eval Time' conversion and 'Type' standardization:")
display(df.head())

DataFrame after 'Eval Time' conversion and 'Type' standardization:


,Claim Id,Samples Received,Type,Eval Time
0,1,TRUE,Standard,11
1,2,TRUE,Standard,11
2,3,TRUE,Standard,9
3,4,TRUE,Standard,8
4,5,TRUE,Standard,8


### Determine Priority Category and Order

Now, we'll create a `Priority_Category` column that reflects your custom sorting and coloring logic. This numerical column will allow us to sort the DataFrame correctly, and then we'll use it to derive the `Priority Order` column with 'High', 'Medium', or 'Low' priority labels.

In [77]:
# Initialize a new column for priority category, which will also act as a sort key
df['Priority_Category'] = 0

# Apply the priority logic based on 'Type' and 'Eval Time'
# Highest Priority (Red) - assigned lowest numbers for ascending sort
df.loc[(df['Type'] == 'NMC') & (df['Eval Time'] > 5), 'Priority_Category'] = 1 # NMC over 5 (Red)
df.loc[(df['Type'] == 'Standard') & (df['Eval Time'] > 10), 'Priority_Category'] = 2 # Standard over 10 (Red)

# Medium Priority (Yellow)
df.loc[(df['Type'] == 'NMC') & (df['Eval Time'] == 5), 'Priority_Category'] = 3 # NMC at 5 (Yellow)
df.loc[(df['Type'] == 'Standard') & (df['Eval Time'] == 10), 'Priority_Category'] = 4 # Standard at 10 (Yellow)

# Lowest Priority (Green) - assigned highest numbers for ascending sort
df.loc[(df['Type'] == 'NMC') & (df['Eval Time'] < 5), 'Priority_Category'] = 5 # NMC under 5 (Green)
df.loc[(df['Type'] == 'Standard') & (df['Eval Time'] < 10), 'Priority_Category'] = 6 # Standard under 10 (Green)

# Sort the DataFrame by 'Priority_Category' (ascending), then by 'Eval Time' (descending)
df_sorted = df.sort_values(by=['Priority_Category', 'Eval Time'], ascending=[True, False]).reset_index(drop=True)


print("DataFrame after creating 'Priority_Category' and sorting:")
display(df_sorted.head())

DataFrame after creating 'Priority_Category' and sorting:


,Claim Id,Samples Received,Type,Eval Time,Priority_Category
0,1,TRUE,Standard,11,2
1,2,TRUE,Standard,11,2
2,12,TRUE,NMC,4,5
3,18,TRUE,NMC,3,5
4,21,TRUE,NMC,3,5


### Create 'Priority Order' Column

Finally, we'll add the 'Priority Order' column based on the `Priority_Category`.

In [78]:
# Create the 'Priority Order' column based on the Priority_Category
def get_priority_label(category):
    if category in [1, 2]:
        return 'High Priority'
    elif category in [3, 4]:
        return 'Medium Priority'
    elif category in [5, 6]:
        return 'Low Priority'
    return 'Unknown'

df_sorted['Priority Order'] = df_sorted['Priority_Category'].apply(get_priority_label)

print("DataFrame with 'Priority Order' column:")
display(df_sorted.head())

DataFrame with 'Priority Order' column:


,Claim Id,Samples Received,Type,Eval Time,Priority_Category,Priority Order
0,1,TRUE,Standard,11,2,High Priority
1,2,TRUE,Standard,11,2,High Priority
2,12,TRUE,NMC,4,5,Low Priority
3,18,TRUE,NMC,3,5,Low Priority
4,21,TRUE,NMC,3,5,Low Priority


In [79]:
# Open the sheet by its URL (the same one used for reading)
sheet = gc.open_by_url(sheet_url)

# Create a new worksheet to write the processed data
try:
    new_worksheet_name = 'Processed Data'
    # Check if a worksheet with this name already exists
    try:
        processed_worksheet = sheet.worksheet(new_worksheet_name)
        # If it exists, clear its content before writing new data
        processed_worksheet.clear()
        print(f"Cleared existing worksheet: '{new_worksheet_name}'")
    except gspread.exceptions.WorksheetNotFound:
        # If it doesn't exist, create it
        processed_worksheet = sheet.add_worksheet(title=new_worksheet_name, rows=df_sorted.shape[0]+1, cols=df_sorted.shape[1]-1) # -1 because we will drop 'Priority_Category'
        print(f"Created new worksheet: '{new_worksheet_name}'")

    # Drop the 'Priority_Category' column as it's for internal sorting and not needed in the final sheet
    df_final = df_sorted.drop(columns=['Priority_Category'])

    # Convert DataFrame to a list of lists, including header
    data_to_write = [df_final.columns.values.tolist()] + df_final.values.tolist()

    # Write the data to the new worksheet
    processed_worksheet.update(data_to_write)

    print(f"Data successfully written to new worksheet: '{new_worksheet_name}'")

except Exception as e:
    print(f"An error occurred while writing to Google Sheets: {e}")

Cleared existing worksheet: 'Processed Data'
Data successfully written to new worksheet: 'Processed Data'


In [80]:
# Apply conditional formatting based on 'Priority Order'
# Define color codes for High, Medium, Low
RED = {'backgroundColor': {'red': 1.0, 'green': 0.0, 'blue': 0.0}}
YELLOW = {'backgroundColor': {'red': 1.0, 'green': 1.0, 'blue': 0.0}}
GREEN = {'backgroundColor': {'red': 0.0, 'green': 1.0, 'blue': 0.0}}

# Get the column index for 'Priority Order' using df_final so it matches the written sheet
priority_col_idx = df_final.columns.get_loc('Priority Order') + 1

# Define the base range for all rules
base_range = {
    'sheetId': processed_worksheet.id,
    'startRowIndex': 1, # Start from the second row (after header)
    'endRowIndex': df_final.shape[0] + 1,
    'startColumnIndex': priority_col_idx - 1,
    'endColumnIndex': priority_col_idx
}

# Define conditional formatting rules (booleanRule part only)
boolean_rules = []

# High Priority (Red)
boolean_rules.append({
    'condition': {
        'type': 'TEXT_EQ',
        'values': [{'userEnteredValue': 'High Priority'}]
    },
    'format': RED
})

# Medium Priority (Yellow)
boolean_rules.append({
    'condition': {
        'type': 'TEXT_EQ',
        'values': [{'userEnteredValue': 'Medium Priority'}]
    },
    'format': YELLOW
})

# Low Priority (Green)
boolean_rules.append({
    'condition': {
        'type': 'TEXT_EQ',
        'values': [{'userEnteredValue': 'Low Priority'}]
    },
    'format': GREEN
})

# Construct the batch update requests
requests = []
for i, boolean_rule in enumerate(boolean_rules):
    requests.append({
        'addConditionalFormatRule': {
            'rule': {
                'ranges': [base_range],
                'booleanRule': boolean_rule
            },
            'index': i # Assign a unique index for each rule
        }
    })

# Send the batch update request to apply formatting
try:
    sheet.batch_update({'requests': requests})
    print("Conditional formatting applied successfully.")
    print(f"You can view the updated sheet here: {sheet_url}")
except Exception as e:
    print(f"An error occurred while applying conditional formatting: {e}")

Conditional formatting applied successfully.
You can view the updated sheet here: https://docs.google.com/spreadsheets/d/1MWQECkBsbjz6AaP7uwCuEFR6xMyzYxy9z6JrOkjGPKM/edit?gid=0#gid=0


### Enhancing Sheet Appearance

In [81]:
print('Applying further aesthetic enhancements to the sheet...')

requests_enhancements = []

# 1. Auto-resize all columns to fit content
# We need to iterate over the columns that were actually written to the sheet (df_final)
# df_final has one less column than df_sorted, as 'Priority_Category' was dropped.
num_final_columns = df_final.shape[1]
for i in range(num_final_columns):
    requests_enhancements.append({
        'autoResizeDimensions': {
            'dimensions': {
                'sheetId': processed_worksheet.id,
                'dimension': 'COLUMNS',
                'startIndex': i,
                'endIndex': i + 1
            }
        }
    })

# 2. Freeze the first header row
requests_enhancements.append({
    'updateSheetProperties': {
        'properties': {
            'sheetId': processed_worksheet.id,
            'gridProperties': {
                'frozenRowCount': 1
            }
        },
        'fields': 'gridProperties.frozenRowCount'
    }
})

# The 'Priority_Category' column is dropped before writing to the sheet (in df_final),
# so there's no need to hide it in the sheet.
# The previous check `if 'Priority_Category' in df.columns:` was checking the original `df`, not `df_final`.

# Send all enhancement requests in a single batch update
try:
    sheet.batch_update({'requests': requests_enhancements})
    print("Sheet appearance enhancements applied successfully.")
    print(f"You can view the updated sheet here: {sheet_url}")
except Exception as e:
    print(f"An error occurred while applying sheet enhancements: {e}")

Applying further aesthetic enhancements to the sheet...
Sheet appearance enhancements applied successfully.
You can view the updated sheet here: https://docs.google.com/spreadsheets/d/1MWQECkBsbjz6AaP7uwCuEFR6xMyzYxy9z6JrOkjGPKM/edit?gid=0#gid=0
